# 3 - Results

This notebook evaluates the trained CNN model on the MNIST test set.
It covers the confusion matrix, classification report, and examples of incorrect predictions.

## 0 - Imports

In [3]:
import sys
import os

sys.path.append(os.path.join(os.path.abspath(""), "..", "src"))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.models import load_model

from data_loader import load_data, preprocess_data

 ## 1 - Data and model loading

In [4]:
(X_train, y_train), (X_test, y_test) = load_data()
(X_train, y_train), (X_test, y_test) = preprocess_data(X_train, y_train, X_test, y_test)

model = load_model("../outputs/models/best_model.keras")
print("Model loaded successfully")

ValueError: File not found: filepath=../outputs/models/best_model.keras. Please ensure the file is an accessible `.keras` zip file.

## 2 - Accuracy on testing set

test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose = 0)

print(f"Test loss : {test_loss:.4f}")
print(f"Test accuracy : {test_accuracy:.4f}")

## 3 - Predictions

In [ ]:
y_pred = model.predict(X_test)
y_pred_cls = np.argmax(y_pred,  axis = 1)
y_true_cls = np.argmax(y_test,  axis = 1)

## 4 - Classification

In [ ]:
print(classification_report(y_true_cls, y_pred_cls))

## 5 - Confusion matrix

In [ ]:
cm = confusion_matrix(y_true_cls, y_pred_cls)

plt.figure(figsize = (10, 8))
sns.heatmap(
    cm,
    annot = True,
    fmt = "d",
    cmap = "Blues",
    xticklabels = range(10),
    yticklabels = range(10)
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.savefig("../outputs/figures/confusion_matrix.png", dpi = 150)
plt.show()

## 6 - Incorrect predictions

In [ ]:
incorrect_idx = np.where(y_pred_cls != y_true_cls)[0]

fig, axes = plt.subplots(3, 5, figsize = (14, 8))
fig.suptitle("Examples of incorrect predictions", fontsize = 14)

for i, ax in enumerate(axes.flat):
    idx = incorrect_idx[i]
    ax.imshow(X_test[idx].reshape(28, 28), cmap = "gray")
    ax.set_title(f"True: {y_true_cls[idx]}  Pred: {y_pred_cls[idx]}", fontsize = 9)
    ax.axis("off")

plt.tight_layout()
plt.savefig("../outputs/figures/incorrect_predictions.png", dpi = 150)
plt.show()

## 7 - Single prediction

In [ ]:
def plot_prediction(idx):
    image      = X_test[idx]
    true_label = y_true_cls[idx]
    pred_label = y_pred_cls[idx]
    confidence = y_pred[idx][pred_label] * 100
    is_correct = true_label == pred_label

    fig, axes = plt.subplots(1, 2, figsize = (10, 4))

    # Image
    axes[0].imshow(image.reshape(28, 28), cmap = "gray")
    color = "green" if is_correct else "red"
    axes[0].set_title(
        f"True: {true_label}  |  Pred: {pred_label}  |  Confidence: {confidence:.1f}%",
        color = color
    )
    axes[0].axis("off")

    # Probability bar chart
    axes[1].bar(range(10), y_pred[idx] * 100, color = "steelblue")
    axes[1].bar(pred_label, y_pred[idx][pred_label] * 100, color = color)
    axes[1].set_title("Predicted probabilities (%)")
    axes[1].set_xlabel("Digit")
    axes[1].set_ylabel("Confidence (%)")
    axes[1].set_xticks(range(10))

    plt.tight_layout()
    plt.show()

# Correct prediction
print("--- Correct prediction ---")
plot_prediction(0)

# Incorrect prediction
print("--- Incorrect prediction ---")
plot_prediction(incorrect_idx[0])

In [ ]:
import tensorflow as tf

def plot_activations(idx):
    image = X_test[idx]

    # Build activation models for both conv layers
    conv1_model = tf.keras.Model(
        inputs  = model.input,
        outputs = model.layers[0].output
    )
    conv2_model = tf.keras.Model(
        inputs  = model.input,
        outputs = model.layers[4].output
    )

    input_img   = image.reshape(1, 28, 28, 1)
    activations_conv1 = conv1_model.predict(input_img, verbose = 0)
    activations_conv2 = conv2_model.predict(input_img, verbose = 0)

    def plot_filters(activations, title, n_filters = 16):
        fig, axes = plt.subplots(2, 8, figsize = (16, 4))
        fig.suptitle(title, fontsize = 13)
        for i, ax in enumerate(axes.flat):
            if i < n_filters:
                ax.imshow(activations[0, :, :, i], cmap = "viridis")
            ax.axis("off")
        plt.tight_layout()
        plt.show()

    # Original image
    plt.figure(figsize = (3, 3))
    plt.imshow(image.reshape(28, 28), cmap = "gray")
    plt.title(f"Original image — True label: {y_true_cls[idx]}")
    plt.axis("off")
    plt.show()

    plot_filters(activations_conv1, "Conv Block 1 — 16 first filters (edges, basic shapes)")
    plot_filters(activations_conv2, "Conv Block 2 — 16 first filters (complex patterns)")

plot_activations(incorrect_idx[0])